## 크로스밸리데이션_회귀예측 

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('house_prices.csv')
df

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48,22.0


In [3]:
features = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX',
       'PTRATIO', 'B', 'LSTAT']

X = df[features]
y = df['MEDV']

In [4]:
# 트레이닝/테스트 데이터 분할
from sklearn.model_selection import train_test_split
X_tn, X_te, y_tn, y_te = train_test_split(X, y, random_state=0)

In [5]:
# 트레이닝/테스트 데이터 분할
from sklearn.model_selection import train_test_split
X_tn, X_te, y_tn, y_te = train_test_split(X, y, random_state=0)

In [6]:
# 데이터 표준화
from sklearn.preprocessing import StandardScaler
std_scale = StandardScaler()
std_scale.fit(X_tn)
X_tn_std = std_scale.transform(X_tn)
X_te_std = std_scale.transform(X_te)

In [7]:
# 그리드 서치 학습
from sklearn import svm
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV

param_grid= {'kernel': ('linear', 'rbf'), 'C': [0.5, 1, 10, 100]}
kfold = KFold(n_splits=5, shuffle=True, random_state=0)
svr = svm.SVR()
grid_cv = GridSearchCV(svr, param_grid, cv=kfold, scoring='neg_mean_squared_error')
grid_cv.fit(X_tn_std, y_tn)

GridSearchCV(cv=KFold(n_splits=5, random_state=0, shuffle=True),
             estimator=SVR(),
             param_grid={'C': [0.5, 1, 10, 100], 'kernel': ('linear', 'rbf')},
             scoring='neg_mean_squared_error')

In [8]:
# 그리드 서치 결과 학인(데이터프레임)
import numpy as np
import pandas as pd

np.transpose (pd.DataFrame(grid_cv.cv_results_))

,0,1,2,3,4,5,6,7
mean_fit_time,0.004331,0.001812,0.003893,0.00257,0.012767,0.004005,0.07877,0.011089
std_fit_time,0.006124,0.000998,0.000219,0.000434,0.003364,0.000004,0.028485,0.006544
mean_score_time,0.000203,0.00232,0.000789,0.001214,0.0006,0.001196,0.0006,0.0002
std_score_time,0.000406,0.001807,0.000396,0.000399,0.00049,0.000401,0.00049,0.0004
param_C,0.5,0.5,1.0,1.0,10.0,10.0,100.0,100.0
param_kernel,linear,rbf,linear,rbf,linear,rbf,linear,rbf
params,"{'C': 0.5, 'kernel': 'linear'}","{'C': 0.5, 'kernel': 'rbf'}","{'C': 1, 'kernel': 'linear'}","{'C': 1, 'kernel': 'rbf'}","{'C': 10, 'kernel': 'linear'}","{'C': 10, 'kernel': 'rbf'}","{'C': 100, 'kernel': 'linear'}","{'C': 100, 'kernel': 'rbf'}"
split0_test_score,-17.445032,-36.17043,-17.386875,-27.228367,-17.458346,-17.409092,-17.446742,-12.562522
split1_test_score,-18.593747,-16.916761,-18.521081,-11.816775,-18.665759,-5.996846,-18.667153,-9.235812
split2_test_score,-40.571874,-76.712963,-38.982199,-57.052392,-37.459318,-26.609204,-37.063074,-15.662599


In [9]:
#베스트 스코어
grid_cv.best_score_

np.float64(-10.185469344281099)

In [10]:
# 베스트 하이퍼파라미터
grid_cv.best_params_

{'C': 100, 'kernel': 'rbf'}

In [13]:
# 최종 모형
model = grid_cv.best_estimator_
print(model)

SVR(C=100)


In [14]:
# 예측
pred_svm = model.predict(X_te_std)
print(pred_svm)

[22.36813301 22.68383242 24.87514322 10.71088583 20.84027476 19.73537584
 23.616982   19.95204534 20.95058207 22.88723384 18.0129821  10.34899842
 14.71816571  5.72642875 47.13543116 31.19225615 23.92516254 36.38402412
 29.8542525  21.3802061  23.4583459  19.83747957 20.28994301 27.14446529
 20.76220706 21.44860078 16.90027694 17.73675701 37.75599541 18.56694782
 14.91270324 17.94677867 20.52063407 19.43864167 25.7174144  20.14369515
  8.93195075 38.44771099 14.22520407 15.35656196 23.40736529 21.70393827
 19.03237972 19.01412226 17.91040447 23.52964267 20.54698635 15.76813636
 15.11317581 22.73652746 19.59564751 25.60790564 22.65165256 41.80916429
 13.06089324 21.4381144  18.26724671 18.22905612 17.33432454 20.41504864
 20.43406878 21.52000315 33.69131932 30.77344561 22.17553542 29.88985454
 15.50819957 19.56480502 14.23453724 23.5303492  20.05641922 21.67942132
 25.81683564 28.05436569 24.93138848  5.38768931 41.99027384 23.13659018
 25.26445049 19.85638849 25.85902651 18.52025113 21

In [15]:
# 모형 평가-MSE
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_te, pred_svm)
print(mse)

17.517111136855807
